# 01 — Bronze : ingestion brute dans le data lake

Objectif : déposer les données téléchargées **sans les modifier** dans le data lake S3 (RustFS),
au format **Delta Lake**, avec la trace de leur provenance.

```
data/raw/bts_2024_MM.csv  (12 fichiers)  ──►  s3a://datalake/bronze/vols       (Delta, partitionné par Month)
data/raw/airports.csv     (OurAirports)  ──►  s3a://datalake/bronze/aeroports  (Delta)
```

Principe de la couche bronze :
- **toutes les colonnes restent en texte** (`string`) : aucun typage, aucun filtre, aucun renommage.
  Le nettoyage et le typage se font dans `02_silver` ;
- on ajoute seulement deux colonnes techniques : `_source_file` (fichier d'origine) et `_ingested_at` (heure d'ingestion) ;
- l'ingestion est **idempotente** : relancer le notebook n'ajoute pas de doublons.

Prérequis : `docker compose up -d` et `bash scripts/telecharger_donnees.sh`.
Console du data lake : http://localhost:9001 (identifiants du `.env`, par défaut rustfsadmin / rustfsadmin).

## 1. Bucket et session Spark

`creer_bucket()` crée le bucket `datalake` s'il n'existe pas. `creer_session()` (dossier `commun/`, repris du cours)
démarre Spark avec Delta Lake et l'accès S3 vers RustFS. Le premier lancement télécharge les jars (quelques minutes).

In [1]:
import sys; sys.path.append("/home/jovyan/work")
import glob
import os

from pyspark.errors import AnalysisException
from pyspark.sql import functions as F
from commun.spark_utils import creer_session, creer_bucket, format_tables, RAW, LAKE

creer_bucket()
spark = creer_session("01_bronze")
assert format_tables(spark) == "delta", (
    "Delta Lake n'est pas actif : lire le message de creer_session(), puis Kernel → Restart.")

BRONZE_VOLS = f"{LAKE}/bronze/vols"
BRONZE_AEROPORTS = f"{LAKE}/bronze/aeroports"

Bucket « datalake » déjà présent.


Préparation de Spark 4.2.0 (Hadoop 3.5.0) + Delta Lake + data lake S3 … le premier démarrage peut prendre quelques minutes.


Session prête. Spark UI : http://localhost:4040


## 2. Les fichiers à ingérer

Un CSV par mois de 2024, produit par `scripts/telecharger_donnees.sh`. Les fichiers BTS se terminent par une virgule
à chaque ligne : Spark voit donc une 110ᵉ colonne vide (nommée `_c109`). En bronze on la **garde** (on ne modifie
pas la source) ; elle sera supprimée en silver.

In [2]:
fichiers = sorted(glob.glob(f"{RAW}/bts_2024_*.csv"))
print(f"{len(fichiers)} fichiers trouvés :")
for f in fichiers:
    print(f"  {os.path.basename(f):<20} {os.path.getsize(f) / 1e6:6.0f} Mo")
assert fichiers, "Aucun CSV : lancer d'abord  bash scripts/telecharger_donnees.sh"

12 fichiers trouvés :
  bts_2024_01.csv         247 Mo
  bts_2024_02.csv         234 Mo
  bts_2024_03.csv         268 Mo
  bts_2024_04.csv         263 Mo
  bts_2024_05.csv         276 Mo
  bts_2024_06.csv         277 Mo
  bts_2024_07.csv         287 Mo
  bts_2024_08.csv         280 Mo
  bts_2024_09.csv         263 Mo
  bts_2024_10.csv         278 Mo
  bts_2024_11.csv         260 Mo
  bts_2024_12.csv         268 Mo


## 3. Les vols → `bronze/vols`

- `inferSchema=False` : Spark lit toutes les colonnes en `string`, sans deviner les types.
- `_metadata.file_name` donne le nom du fichier lu pour chaque ligne (colonne `_source_file`).
- La table est **partitionnée par `Month`** : un dossier par mois dans RustFS. Ce n'est pas une transformation
  des données, seulement un rangement des fichiers ; il permet de relire ou de remplacer un mois sans parcourir l'année.

**Idempotence** : on ne lit que les fichiers absents de la table (comparaison sur `_source_file`), puis on les ajoute
en `append`. Au premier lancement, tout est ingéré ; si on relance, rien n'est ajouté ; si on télécharge un mois de
plus, seul ce mois est ajouté.

In [3]:
def lire_csv_brut(chemins):
    """Lit des CSV BTS en texte brut, avec les deux colonnes techniques."""
    return (spark.read
            .option("header", True)
            .option("inferSchema", False)
            .csv(chemins)
            .withColumn("_source_file", F.col("_metadata.file_name"))
            .withColumn("_ingested_at", F.current_timestamp()))


def deja_ingeres():
    """Noms des fichiers déjà présents dans bronze/vols (vide si la table n'existe pas)."""
    try:
        lignes = spark.read.format("delta").load(BRONZE_VOLS).select("_source_file").distinct().collect()
        return {r["_source_file"] for r in lignes}
    except AnalysisException:   # la table n'existe pas encore
        return set()


presents = deja_ingeres()
a_ingerer = [f for f in fichiers if os.path.basename(f) not in presents]
print(f"Déjà dans le bronze : {len(presents)} fichier(s) | à ingérer : {len(a_ingerer)}")

if a_ingerer:
    (lire_csv_brut(a_ingerer)
        .write.format("delta")
        .mode("append")
        .partitionBy("Month")
        .save(BRONZE_VOLS))
    print("Écriture terminée.")
else:
    print("Rien à ajouter : le bronze est à jour.")

Déjà dans le bronze : 0 fichier(s) | à ingérer : 12


Écriture terminée.


### Contrôle : lignes par fichier

On vérifie que chaque fichier est présent une seule fois et que le total dépasse les **2 millions de lignes**
demandés par l'énoncé.

In [4]:
vols = spark.read.format("delta").load(BRONZE_VOLS)

par_fichier = vols.groupBy("_source_file").count().orderBy("_source_file")
par_fichier.show(20, truncate=False)

total = vols.count()
print(f"Total bronze/vols : {total:,} lignes, {len(vols.columns)} colonnes".replace(",", " "))
assert total >= 2_000_000, "Moins de 2 M de lignes : vérifier le téléchargement."

+---------------+------+
|_source_file   |count |
+---------------+------+
|bts_2024_01.csv|547271|
|bts_2024_02.csv|519221|
|bts_2024_03.csv|591767|
|bts_2024_04.csv|582185|
|bts_2024_05.csv|609743|
|bts_2024_06.csv|611132|
|bts_2024_07.csv|634613|
|bts_2024_08.csv|619025|
|bts_2024_09.csv|582622|
|bts_2024_10.csv|615497|
|bts_2024_11.csv|575404|
|bts_2024_12.csv|590581|
+---------------+------+



Total bronze/vols : 7 079 061 lignes  112 colonnes


In [5]:
# Toutes les colonnes sont bien du texte (sauf la colonne technique _ingested_at)
types = {t for c, t in vols.dtypes if c != "_ingested_at"}
print("Types des colonnes métier :", types)
vols.select("FlightDate", "Reporting_Airline", "Origin", "Dest", "DepDelay", "ArrDelay", "Cancelled", "_source_file").show(5)

Types des colonnes métier : {'string'}


+----------+-----------------+------+----+--------+--------+---------+---------------+
|FlightDate|Reporting_Airline|Origin|Dest|DepDelay|ArrDelay|Cancelled|   _source_file|
+----------+-----------------+------+----+--------+--------+---------+---------------+
|2024-09-16|               9E|   CVG| LGA|   -8.00|  -16.00|     0.00|bts_2024_09.csv|
|2024-09-17|               9E|   CVG| LGA|   -5.00|   -6.00|     0.00|bts_2024_09.csv|
|2024-09-18|               9E|   CVG| LGA|   -5.00|    5.00|     0.00|bts_2024_09.csv|
|2024-09-19|               9E|   CVG| LGA|   -9.00|  -14.00|     0.00|bts_2024_09.csv|
|2024-09-20|               9E|   CVG| LGA|   -7.00|  -15.00|     0.00|bts_2024_09.csv|
+----------+-----------------+------+----+--------+--------+---------+---------------+
only showing top 5 rows


## 4. Le référentiel des aéroports → `bronze/aeroports`

`airports.csv` d'OurAirports (environ 80 000 aérodromes dans le monde) apporte les coordonnées (`latitude_deg`,
`longitude_deg`) nécessaires à la carte. On l'ingère brut, lui aussi tout en texte : le filtre sur les aéroports
américains et la jointure sur le code IATA se feront en silver.

Ce fichier est un référentiel qu'on remplace en entier à chaque version : on écrit donc en `overwrite`.

In [6]:
aeroports = (spark.read
             .option("header", True)
             .option("inferSchema", False)
             .option("multiLine", True)      # certains noms contiennent des retours à la ligne entre guillemets
             .option("escape", '"')
             .csv(f"{RAW}/airports.csv")
             .withColumn("_source_file", F.col("_metadata.file_name"))
             .withColumn("_ingested_at", F.current_timestamp()))

aeroports.write.format("delta").mode("overwrite").save(BRONZE_AEROPORTS)

aeroports = spark.read.format("delta").load(BRONZE_AEROPORTS)
print(f"bronze/aeroports : {aeroports.count():,} lignes".replace(",", " "))
aeroports.select("ident", "iata_code", "name", "iso_country", "latitude_deg", "longitude_deg").show(5, truncate=False)

bronze/aeroports : 86 387 lignes


+-----+---------+--------------------+-----------+-----------------+------------------+
|ident|iata_code|name                |iso_country|latitude_deg     |longitude_deg     |
+-----+---------+--------------------+-----------+-----------------+------------------+
|00A  |NULL     |Total RF Heliport   |US         |40.070985        |-74.933689        |
|00AA |NULL     |Aero B Ranch Airport|US         |38.704022        |-101.473911       |
|00AK |NULL     |Lowell Field        |US         |59.947733        |-151.692524       |
|00AL |NULL     |Epps Airpark        |US         |34.86479949951172|-86.77030181884766|
|00AN |NULL     |Katmai Lodge Airport|US         |59.093287        |-156.456699       |
+-----+---------+--------------------+-----------+-----------------+------------------+
only showing top 5 rows


Dans la console RustFS, `bronze/vols` contient un dossier par mois (`Month=1`, `Month=2`…) et un dossier
`_delta_log` : c'est le **journal des transactions** Delta, qui rend possibles les démonstrations suivantes.

## 5. Pourquoi Delta Lake plutôt que du Parquet simple ?

### a) Historique et *time travel*

Chaque écriture crée une nouvelle **version** de la table. Pour le montrer, on supprime volontairement le mois de
janvier (version suivante), on relit l'ancienne version, puis on restaure. Comme la table est partitionnée par mois,
la suppression ne réécrit aucun fichier : Delta marque simplement ceux de `Month=1` comme retirés dans son journal.

In [7]:
from delta.tables import DeltaTable

table_vols = DeltaTable.forPath(spark, BRONZE_VOLS)
avant = spark.read.format("delta").load(BRONZE_VOLS).count()

table_vols.delete("Month = '1'")          # nouvelle version, sans janvier
apres = spark.read.format("delta").load(BRONZE_VOLS).count()
print(f"Avant : {avant:,} lignes | après suppression de janvier : {apres:,}".replace(",", " "))

Avant : 7 079 061 lignes | après suppression de janvier : 6 531 790


In [8]:
spark.sql(f"DESCRIBE HISTORY delta.`{BRONZE_VOLS}`") \
     .select("version", "timestamp", "operation", "operationParameters").show(truncate=60)

+-------+-------------------+---------+------------------------------------------+
|version|          timestamp|operation|                       operationParameters|
+-------+-------------------+---------+------------------------------------------+
|      1|2026-11-24 07:22:19|   DELETE|       {predicate -> ["(Month#7203 = 1)"]}|
|      0|2026-11-24 07:22:04|    WRITE|{mode -> Append, partitionBy -> ["Month"]}|
+-------+-------------------+---------+------------------------------------------+



In [9]:
# Time travel : on relit la table telle qu'elle était juste avant la suppression
version_supprimee = table_vols.history(1).select("version").first()["version"]
version_avant = version_supprimee - 1

relue = spark.read.format("delta").option("versionAsOf", version_avant).load(BRONZE_VOLS).count()
print(f"Version {version_avant} relue : {relue:,} lignes".replace(",", " "))

# Le bronze doit rester complet : on restaure cette version
table_vols.restoreToVersion(version_avant)
restauree = spark.read.format("delta").load(BRONZE_VOLS).count()
print(f"Après restauration : {restauree:,} lignes".replace(",", " "))
assert restauree == avant

Version 0 relue : 7 079 061 lignes


Après restauration : 7 079 061 lignes


### b) Contrôle du schéma

Delta refuse une écriture dont les colonnes ne correspondent pas à celles de la table : une erreur d'ingestion ne
peut pas corrompre le bronze en silence.

In [10]:
intrus = spark.createDataFrame([("FAKE", "test")], ["iata_code", "colonne_inconnue"])
try:
    intrus.write.format("delta").mode("append").save(BRONZE_AEROPORTS)
    print("⚠ Écriture acceptée : ce n'était pas attendu.")
except Exception as e:
    print("Écriture refusée par Delta :", str(e).splitlines()[0][:200])

Écriture refusée par Delta : [DELTA_METADATA_MISMATCH] A metadata mismatch was detected when writing to the Delta table.


### c) Ajout d'un mois sans doublon

Relancer la cellule d'ingestion de la section 3 affiche « Rien à ajouter » : chaque fichier n'est ingéré qu'une fois.
Pour ajouter des données (par exemple 2025), il suffit de télécharger les fichiers puis de relancer ce notebook.

## C'est réussi quand…

- la console RustFS montre `bronze/vols` (un dossier par mois) et `bronze/aeroports` ;
- le contrôle affiche 12 fichiers et plus de 2 M de lignes ;
- l'historique Delta montre l'écriture, la suppression et la restauration, et la table a retrouvé toutes ses lignes.

In [11]:
spark.stop()